In [1]:
# %%

#------------------------------------------------ Begin_Librairie ----------------------------------------

from bs4 import BeautifulSoup

import datetime

import pandas as pd

import numpy as np

from pandas import ExcelWriter

from time import sleep

import os

import re

import tabula

from tabula.io import read_pdf

from selenium import webdriver

from selenium.webdriver.common.by import By

import pdfplumber



In [2]:

# %%

#------------------------------------------------ Begin_ fileName ----------------------------------------

regulatorName = 'MX CNSF' ## change to current controller name



print(f"Running {regulatorName} Web Scraping Tool v.1.2")

now=datetime.datetime.now()

filename = '{} SQL Read {}.xlsx'.format(regulatorName, str(now).replace(":",".")[:-7])


scriptfolder = f"C:\\Users\\wuj1\\OneDrive - Moody's\\Desktop\\Regulator\\{regulatorName}"
#scriptfolder=os.path.dirname(os.path.abspath(__file__)) ## to decomment for the production environment

os.chdir(scriptfolder)

writer = ExcelWriter(filename)

tempfolder=os.path.join(scriptfolder, 'tempfolder') #if files are downloaded during the process



if os.path.exists(tempfolder):

    for rem in os.listdir(tempfolder):

        os.remove(os.path.join(tempfolder, rem))

else:

    os.mkdir(tempfolder)



Running MX CNSF Web Scraping Tool v.1.2


In [3]:


# %%

#------------------------------------------------ Begin_chromedriver ----------------------------------------

#Starting Chrome driver, set to download files in tempfolder

chromeOptions = webdriver.ChromeOptions()

prefs = {"plugins.always_open_pdf_externally": True,

		 "download.prompt_for_download": False,

		 "download.default_directory" : tempfolder}

chromeOptions.add_experimental_option("prefs",prefs)

driver = webdriver.Chrome(options=chromeOptions)

driver.maximize_window()



In [4]:


# %%

#------------------------------------------------ Begin_variable----------------------------------------

regdict={

        'MX CNSF 2': "//a[contains (@onclick,'Oficinas de Representación')]", 

        'MX CNSF 3': "//a[contains (@onclick,'Registro General de Reaseguradoras Extranjeras')]"

         }



sqldict = {'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [], 'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [],

          'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [], 'Address_1': [], 'Address_2': [], 'City': [],

          'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [],

          'RegCtry': [], 'RegCode' : [], 'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],

          'Address_1 - Mother company': [], 'Address_2 -  Mother company': [], 'City - Mother company': [], 'Zip - Mother company': [], 'Cntry - Mother company': [],

          'Phone - Mother company': [], 'Check': []}



ISO_MX={"EspaÃ±a" :"ES", "E.U.A." :"US", "Alemania" :"DE","Bermuda" :"BM",

        "Francia" :"FR","PanamÃ¡" :"PA","Unido." :"GB", " México": "MX", " Mexico": "MX"}



columns2=["Index", "NAME", "No. de Registro",'REPRESENTANTE LEGAL','DOMICILIO']

columns3=['Index','NAME', 'No. de Registro', 'Operación']

Typology={ regulatorName+' 2': 'Oficinas de representación de Reaseguradores extranjeros inscritas en el registro general de reaseguradoras extranjeras (RGRE)',
          regulatorName+' 3': 'Reaseguradoras inscritas en el Registro General de Reaseguradoras Extranjeras para tomar reaseguro y reafianzamiento del país',
          }

processdate = now.strftime('%Y-%m-%d')




In [5]:

# %%

#------------------------------------------------ Begin_Fouction ----------------------------------------

def bourange_same_length_array(sqldict) :

    len_value=[]

    for key, value in sqldict.items():

        len_value.append(len(value))

    maxlen = max(len_value)

    for key, val in sqldict.items():

        if len(sqldict[key]) != maxlen:

            empty = []

            total_empty = maxlen - len(sqldict[key])

            for i in range(total_empty):

                empty.append('')

            sqldict[key]=sqldict[key]+empty

    return sqldict






In [6]:

# %%

#------------------------------------------------ Begin_Main ----------------------------------------

driver.get('https://www.gob.mx/cnsf/documentos/reaseguradores-extranjeros-e-intermediarios')

for reg in regdict:

    print(f'[INFO] : Working with {reg}')


    link_cnsf= driver.find_element(By.XPATH,f"{regdict[reg]}")

    file_cnsf= link_cnsf.get_attribute('href')

    print(file_cnsf)

    driver.get(file_cnsf)

    sleep(3)
    # cnsf = tabula.io.read_pdf(file_cnsf,lattice=True,pages='all',pandas_options={'dtype':str,'header':None})
    dl_files = [os.path.join(tempfolder, f) for f in os.listdir(tempfolder)]
    if reg == 'MX CNSF 2' :
        with pdfplumber.open(dl_files[0]) as pdf:
            for page in pdf.pages:
                page_tables = page.extract_tables()    

                for info in page_tables[:][0][:]:
                    try:
                        if info[0].isdigit():

                            phone = re.search(r"(\d{2}[\W]*?\d{2}[\W]*?\d{2}[\W]*?\d{2}[\W]*?\d{2})", info[4])
                            phone = phone.group(1) if phone else ''
                            zip_match = re.search(r'(\d{5})', info[4])
                            zip_code = zip_match.group(1).zfill(5) if zip_match else ''
                            

                            email = re.search(r'(\S+@\S+)', info[4])
                            email = email.group(1) if email else ''
                            #print('email:', email)
                            if email.split(':'):
                                sqldict['Email'].append(email.split(':')[-1].strip())
                            else:
                                sqldict['Email'].append(email)
                            
                            sqldict['Name'].append(info[1])
                            sqldict['InternalID_1'].append(info[2])
                            sqldict['InternalID_1_type'].append('No. de Registro')
                            sqldict['RegulationType'].append('Regulated')
                            sqldict['ListProcessDate'].append(processdate)
                            sqldict['RegCtry'].append(reg.split(' ')[0])
                            sqldict['RegCode'].append(reg.split(' ')[1])
                            sqldict['ListCode'].append(reg.split(' ')[-1])
                            sqldict['Address_1'].append(info[4].split('\n')[0])
                            sqldict['Phone'].append( phone.replace('\n',''))
                            sqldict['Zip'].append(zip_code)
                            sqldict['License_Type'].append('')
                            sqldict['ListName'].append(Typology[reg])
                            sqldict = bourange_same_length_array(sqldict)

                    except:
                        pass

    elif reg == 'MX CNSF 3' :
        with pdfplumber.open(dl_files[0]) as pdf:
            for page in pdf.pages:
                page_tables = page.extract_tables()
                if len(page_tables[0][0]) >2:
                    #print(len(page_tables[0]))
                    for info in page_tables[:][0][:]:
                        try:
                            if info[0].isdigit():

                                sqldict['Name'].append(info[1])
                                sqldict['InternalID_1'].append(info[2])
                                sqldict['InternalID_1_type'].append('No. de Registro')
                                sqldict['RegulationType'].append('Regulated')
                                sqldict['ListProcessDate'].append(processdate)
                                sqldict['RegCtry'].append(reg.split(' ')[0])
                                sqldict['RegCode'].append(reg.split(' ')[1])
                                sqldict['ListCode'].append(reg.split(' ')[-1])
                                sqldict['ListName'].append(Typology[reg])
                                sqldict['License_Type'].append('')


                
                        except:
                            pass

                if len(page_tables) >1 or len(page_tables[0][0]) <4  :
                    
                    for i in page_tables[::]:
                        for info in i:
                            if info[0].isdigit() and int(info[0]) >100:
                                flag = info[1].split('RGRE')[0].replace('\n','').strip()
                                rgre_id = info[1].split('RGRE')[-1].replace('\n','').strip()
                            else:

                                name =  info[1]
                                if name is not None and name !='':
                                    sqldict['Name'].append(name)
                                    sqldict['InternalID_1'].append('RGRE'+rgre_id)
                                    sqldict['InternalID_1_type'].append('No. de Registro')
                                    sqldict['RegulationType'].append('Regulated')
                                    sqldict['ListProcessDate'].append(processdate)
                                    sqldict['RegCtry'].append(reg.split(' ')[0])
                                    sqldict['RegCode'].append(reg.split(' ')[1])
                                    sqldict['ListCode'].append(reg.split(' ')[-1])
                                    sqldict['ListName'].append(Typology[reg])
                                    sqldict['License_Type'].append(flag)
                                    sqldict = bourange_same_length_array(sqldict)
        
        
    sqldict = bourange_same_length_array(sqldict)

    sleep(1)





[INFO] : Working with MX CNSF 2
https://www.gob.mx/cms/uploads/attachment/file/1017315/R-DGSR-002C-OFICINAS-JULIO_2025.pdf
[INFO] : Working with MX CNSF 3
https://www.gob.mx/cms/uploads/attachment/file/1017314/R-DGSR-002A-REASEG-JULIO_2025.pdf


In [7]:


# %%

#------------------------------------------------ Begin_writer and save df to excel  ----------------------------------------

os.chdir(scriptfolder)

df=pd.DataFrame(sqldict)

# Supprimer les lignes où la colonne 'Name' est vide

# df = df[df['Name'] != '']

df['Zip'] = df['Zip'].astype(str) 

df.to_excel(writer, 'SQL Ready', index=False)

writer.save()

writer.close()

driver.quit()

sleep(3)
    

C:\Users\wuj1\AppData\Local\Temp\1\ipykernel_31504\3701412476.py:15: FutureWarning: Starting with pandas version 3.0 all arguments of to_excel except for the argument 'excel_writer' will be keyword-only.
  df.to_excel(writer, 'SQL Ready', index=False)


AttributeError: 'OpenpyxlWriter' object has no attribute 'save'

In [9]:
df.to_csv('total_v9.csv')